# CodeCloneValidation

In [7]:
from datasets import load_dataset

# Load the dataset
ds = load_dataset("google/code_x_glue_cc_clone_detection_big_clone_bench")

In [8]:
# Display the dataset
print(ds)

DatasetDict({
    train: Dataset({
        features: ['id', 'id1', 'id2', 'func1', 'func2', 'label'],
        num_rows: 901028
    })
    validation: Dataset({
        features: ['id', 'id1', 'id2', 'func1', 'func2', 'label'],
        num_rows: 415416
    })
    test: Dataset({
        features: ['id', 'id1', 'id2', 'func1', 'func2', 'label'],
        num_rows: 415416
    })
})


# First experience (Tree of decision)

In [23]:
import numpy as np
import scipy.sparse
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

TRAIN_SIZE = 100000
TEST_SIZE = 50000

# Split the dataset
train_data = ds['train'][:TRAIN_SIZE]
test_data = ds['test'][:TEST_SIZE]

# Vectorization (we limit the number of features for performance)
vectorizer = TfidfVectorizer(max_features=5000)

# Fit the vectorizer on the training code snippets
all_train_code = train_data['func1'] + train_data['func2']
vectorizer.fit(all_train_code)

# Transform the training code snippets
X_train_f1 = vectorizer.transform(train_data['func1'])
X_train_f2 = vectorizer.transform(train_data['func2'])

# Transform the test code snippets
X_test_f1 = vectorizer.transform(test_data['func1'])
X_test_f2 = vectorizer.transform(test_data['func2'])

# Create feature vectors by computing the absolute difference
X_train = scipy.sparse.csr_matrix(abs(X_train_f1 - X_train_f2))
X_test = scipy.sparse.csr_matrix(abs(X_test_f1 - X_test_f2))

y_train = train_data['label']
y_test = test_data['label']

# Train the Decision Tree Classifier
clf = DecisionTreeClassifier(max_depth=10, random_state=42)
clf.fit(X_train, y_train)

# Evaluate the model
y_pred = clf.predict(X_test)
print("Accuracy:", accuracy_score(y_test, y_pred))



Accuracy: 0.93806
